Imports:

In [51]:
import aerosandbox as asb
import aerosandbox.numpy as np
import casadi as ca

from wing_deflection import max_deflection

Constants, definitions that shouldn't change

In [52]:
wing_airfoil = asb.Airfoil("naca0001")
tail_airfoil = asb.Airfoil("naca0001")

E = 3e9
rho_balsa = 250 # kg/m3
rho = 1.225
nu = 1.5e-5
pi = np.pi
AR_v = 3


Beginning the optimization environment, initializing some variables


In [53]:
opti = asb.Opti()

V = opti.variable(init_guess = 4, lower_bound = 0.1, upper_bound = 7)
AR = opti.variable(init_guess = 6, lower_bound = 5, upper_bound = 15)
S = opti.variable(init_guess = 0.05, lower_bound = 0.01, upper_bound = 0.5)
twist = opti.variable(init_guess = 2, lower_bound = 0, upper_bound = 10)
dihedral = opti.variable(init_guess = 5, lower_bound = 0, upper_bound=10)
AR_h = opti.variable(init_guess = 4, lower_bound = 2, upper_bound =10)
S_h = opti.variable(init_guess = 0.01, lower_bound = 0.00001)
S_v = opti.variable(init_guess = 0.005, lower_bound = 0.000001)
glide_angle = opti.variable(init_guess = 5, lower_bound=0, upper_bound = 15)
H_loc = opti.variable(init_guess = 0.2, lower_bound = 0) #horizontal stabilizer is at H_loc
H_loc_v = 0


Derived variables from the given optimized variables

In [54]:
b_total = ca.sqrt(S * AR)
c = S / b_total
b = b_total / 2

b_h = ca.sqrt(S_h * AR_h) / 2
c_h = S_h / (b_h * 2)

b_v = ca.sqrt(S_v * AR_v)
c_v = S_v / b_v

Weight and structural model

In [55]:
thickness = 0.002
boom_area = 0.01 * 0.01
margin = 0.01
I_formula = lambda chord, tau: chord * thickness ** 3 / 12

weight = (S + S_h + S_v) * thickness * rho_balsa + H_loc * rho_balsa * boom_area + margin

cumsum_weight = (c / 2 * S + (H_loc + c_h/2) * S_h + (H_loc + c_v/2) * S_v) * thickness * rho_balsa + H_loc / 2 * H_loc * boom_area * rho_balsa 
COM = cumsum_weight / weight

# Structural Deflection Constraint (tau set to 0.05 dynamically)
N = 1.2
opti.subject_to(max_deflection(N * weight * 9.81, AR, S, E, I_formula = I_formula, tau=0.002) < 0.1 * b)
opti.subject_to(max_deflection(N * weight * 9.81, AR_h, S_h, E, I_formula = I_formula, tau=0.002) < 0.1 * b_h)
# opti.subject_to(max_deflection(N * weight * 9.81, AR_v, S_v, E, I_formula = I_formula, tau=0.05) < 0.02)

opti.subject_to(H_loc > c)

MX(fabs(opti2_lam_g_20))

Aerodynamic Constraints

In [56]:
import casadi as ca

#tail volume coefficients

l_h = H_loc - c / 4 + c_h / 4
l_v = H_loc - c / 4 + c_v / 4

hor_vol_coef = S_h * l_h / (S * c)
ver_vol_coef = S_v * l_v / (S * c)

opti.subject_to(hor_vol_coef > 0.3)
opti.subject_to(hor_vol_coef < 0.6)

opti.subject_to(ver_vol_coef > 0.02)
opti.subject_to(ver_vol_coef < 0.05)

#neutral point
a_w = 2 * pi / (1 + 2 / AR)
a_h = 2 * pi / (1 + 2 / AR_h)

# np = c * (a_w / (4 * a_h) + hor_vol_coef * (1 + c / (4 * l_h))) / (a_w / a_h + hor_vol_coef * c / l_h )
np = c * (S * a_w / (4 * S_h * a_h) + l_h / c + 0.25) / (S * a_w / (S_h * a_h) + 1)

#recalculate neutral point and stuff
opti.subject_to(np > COM)

#aerodynamic constraints

a0 = 2 * pi
e = 0.9
Re = V * c / nu

CL = a0 / (1 + a0 / (pi * AR * e)) * twist

# CL_h = - (COM / c - 0.25) * CL / ((COM / H_loc - 1 - c / H_loc) * hor_vol_coef)
CL_h = (CL * (COM - c/4)) / l_h

alpha_t = CL_h / (2 * pi)

CDi = CL ** 2 / (pi * AR * e) + CL_h ** 2 / (pi * AR_h * e)
CD0 = 1.3 * 2.656 / Re ** 0.5
CD = CDi + CD0

q = 1/2 * rho * V**2 * S
q_h = 1/2 * rho * V**2 * S_h
L = q * (CL) + q_h * CL_h
D = q * CD

opti.subject_to(weight * 9.81 * ca.cos(glide_angle * pi / 180) == L)
# opti.subject_to(weight * 9.81 * ca.cos(glide_angle * pi / 180) > L - 0.01)

opti.subject_to(weight * 9.81 * ca.sin(glide_angle * pi / 180) == D)
# opti.subject_to(weight * 9.81 * ca.sin(glide_angle * pi / 180) < D + 0.01)

opti.subject_to(CL < 1.4)
opti.subject_to(CL_h < 1.4)

#spiral stability
opti.subject_to((l_v - c_v / 4) * dihedral > 5 * b_total * CL)

MX(fabs(opti2_lam_g_30))

Objective function and solving

In [57]:
total_dist = 1 / (ca.sin(glide_angle))


opti.maximize(total_dist / V)


# --- Solve the Problem ---
sol = opti.solve(behavior_on_failure="return_last", max_iter=10000)


This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:       18
Number of nonzeros in inequality constraint Jacobian.:       69
Number of nonzeros in Lagrangian Hessian.............:       46

Total number of variables............................:       10
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        2
Total number of inequality constraints...............:       28
        inequality constraints with only lower bounds:       12
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:       16

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  2.6070880e-01 2.42e+01 1.03e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

Output statements

In [58]:
print("\n" + "="*40)
print("       Grass'S GLIDER DESIGN METRICS     ")
print("="*40)
print(f"Optimization Status : {sol.stats()['return_status']}")
print(f"Total Glider Mass   : {sol.value(weight) * 1000:.2f} grams")
print(f"Design Velocity     : {sol.value(V):.2f} m/s")
print("-"*40)
print(f"Main Wing Area (S)  : {sol.value(S):.4f} m^2")
print(f"Aspect Ratio (AR)   : {sol.value(AR):.2f}")
print(f"Total Wingspan (b)  : {sol.value(b_total):.2f} m")
print(f"Wing Chord (c)      : {sol.value(c)*1000:.1f} mm")
print(f"Wing Dihedral       : {sol.value(dihedral):.1f} degrees")
print(f"Wing Twist          : {sol.value(twist):.4f} degrees")
print(f"Glide Angle          : {sol.value(glide_angle):.1f} degrees")
print("-"*40)
print(f"Horizontal Tail Area: {sol.value(S_h):.4f} m^2")
print(f"Vertical Tail Area  : {sol.value(S_v):.4f} m^2")

print(f"Horizontal Tail AR: {sol.value(AR_h):.4f} m^2")
print(f"Horizontal Tail b: {sol.value(b_h * 2):.4f} m")
print(f"Horizontal Tail c: {sol.value(c_h):.4f} m")

print(f"Vertical Tail AR: {sol.value(AR_v):.4f} m^2")
print(f"Vertical Tail b: {sol.value(b_v):.4f} m")
print(f"Vertical Tail c: {sol.value(c_v):.4f} m")

print(f"Tail Boom Length    : {sol.value(H_loc):.2f} m")
print(f"Tail Boom Length    : {sol.value(H_loc_v):.2f} m")
print(f"decalage    : {sol.value(alpha_t):.4f} degrees")

print("-"*40)
print(f"Aerodynamic Lift    : {sol.value(L):.3f} N")
print(f"Required Weight Lift: {sol.value(weight)*9.81:.3f} N")
print(f"Operating CL        : {sol.value(CL):.3f}")
print("-"*40)
print(f"Horiz. Vol. Coeff.  : {sol.value(hor_vol_coef):.3f}  (Target: 0.3 - 0.6)")
print(f"Vert. Vol. Coeff.   : {sol.value(ver_vol_coef):.3f}  (Target: 0.02 - 0.05)")
print(f"Spiral Criterion (B): {sol.value(spiral):.3f}")
# print(sol.value(CL_h))
# print(sol.value(CL))
# print(sol.value(twist))
# print(sol.value( a0 / (1 + a0 / (pi * AR * e)) * twist))

# print(sol.value(max_deflection(N * weight * 9.81, AR_h, S_h, E, I_formula = I_formula, tau=0.002)))
# print(sol.value(b_h))

# print(sol.value(np))
# print(sol.value(COM))


       Grass'S GLIDER DESIGN METRICS     
Optimization Status : Solve_Succeeded
Total Glider Mass   : 19.69 grams
Design Velocity     : 7.00 m/s
----------------------------------------
Main Wing Area (S)  : 0.0100 m^2
Aspect Ratio (AR)   : 5.00
Total Wingspan (b)  : 0.22 m
Wing Chord (c)      : 44.7 mm
Wing Dihedral       : 6.4 degrees
Wing Twist          : 0.1444 degrees
Glide Angle          : 4.7 degrees
----------------------------------------
Horizontal Tail Area: 0.0018 m^2
Vertical Tail Area  : 0.0001 m^2
Horizontal Tail AR: 2.0000 m^2
Horizontal Tail b: 0.0606 m
Horizontal Tail c: 0.0303 m
Vertical Tail AR: 3.0000 m^2
Vertical Tail b: 0.0139 m
Vertical Tail c: 0.0046 m
Tail Boom Length    : 0.15 m
Tail Boom Length    : 0.00 m
decalage    : 0.0114 degrees
----------------------------------------
Aerodynamic Lift    : 0.193 N
Required Weight Lift: 0.193 N
Operating CL        : 0.628
----------------------------------------
Horiz. Vol. Coeff.  : 0.600  (Target: 0.3 - 0.6)
Vert. V

RuntimeError: Error in Opti::value [OptiNode] at .../casadi/core/optistack.cpp:233:
.../casadi/core/optistack_internal.cpp:653: Unknown: Opti decision variable 'opti1_x_10' of shape 1x1, belonging to a different instance of Opti.

In [39]:
airplane = asb.Airplane(
    name="Peter's Glider",
    xyz_ref=[0, 0, 0],  
    wings=[
        asb.Wing(
            name="Main Wing",
            symmetric=True,  
            xsecs=[  
                asb.WingXSec(  
                    xyz_le=[0, 0, 0],  
                    chord=c,
                    twist=twist,  
                    airfoil=wing_airfoil,  
                ),
                asb.WingXSec(  
                    xyz_le=[0, b, b * dihedral * pi / 180],
                    chord=c,
                    twist=twist,
                    airfoil=wing_airfoil,
                ),
            ],
        ),
        asb.Wing(
            name="Horizontal Stabilizer",
            symmetric=True,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_h,
                    twist=alpha_t,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, b_h, 0], chord=c_h, twist=0, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc, 0, 0]),
        asb.Wing(
            name="Vertical Stabilizer",
            symmetric=False,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_v,
                    twist=0,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, 0, b_v], chord=c_v, twist=0, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc, 0, 0]),
    ],
)

optimized_airplane = sol.value(airplane)

# This will open up an interactive browser window showing your full 3D layout!
optimized_airplane.draw()

Widget(value='<iframe src="http://localhost:35307/index.html?ui=P_0x722281387920_4&reconnect=auto" class="pyvi…

PolyData (0x7222813478e0)
  N Cells:    725
  N Points:   730
  N Strips:   0
  X Bounds:   -2.239e-25, 7.789e-02
  Y Bounds:   -1.373e-01, 1.373e-01
  Z Bounds:   -2.910e-04, 3.359e-02
  N Arrays:   0

In [40]:
opti.debug.show_infeasibilities()




Violated constraints (tol 0), in order of declaration:
------- i = 1/79 ------ 
-inf <= 1.75 <= 1.75 (viol 1.74865e-08)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in subject_to
  super().subject_to(constraint)
------- i = 2/79 ------ 
0.833333 <= 0.833333 <= inf (viol 9.97623e-09)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in subject_to
  super().subject_to(constraint)
------- i = 9/79 ------ 
-inf <= 2 <= 2 (viol 1.9967e-08)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in subject_to
  super().subject_to(constraint)
------- i = 11/79 ------ 
-inf <= 2.5 <= 2.5 (viol 2.47321e-08)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in su